In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import sys
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.screener.engine import load_screener_data, run_all_presets

# Load screener data
screener_df = load_screener_data()

print(f"Screener data loaded: {screener_df.shape}")
print(f"Companies: {screener_df['company_id'].nunique()}")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Screener data loaded: (98, 50)
Companies: 98


In [2]:
def winsorise(series, p_low=10, p_high=90):
    """
    Caps extreme values at P10 and P90 percentiles.
    Prevents outliers like BEL/HAL from distorting scores.

    Args:
        series: Pandas Series of numeric values
        p_low:  Lower percentile cap (default 10)
        p_high: Upper percentile cap (default 90)

    Returns:
        Series: Winsorised values
    """
    low  = series.quantile(p_low / 100)
    high = series.quantile(p_high / 100)
    return series.clip(lower=low, upper=high)

def scale_to_100(series):
    """
    Scales a series to 0-100 range using min-max normalisation.
    After winsorisation, min maps to 0 and max maps to 100.
    """
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([50.0] * len(series), index=series.index)
    return ((series - min_val) / (max_val - min_val) * 100).round(2)

# Test winsorisation
test = pd.Series([1, 2, 3, 4, 5, 1000])
print("Before winsorisation:", test.tolist())
print("After winsorisation: ", winsorise(test).tolist())

Before winsorisation: [1, 2, 3, 4, 5, 1000]
After winsorisation:  [1.5, 2.0, 3.0, 4.0, 5.0, 502.5]


In [4]:
def compute_health_score(df):
    """
    Computes Financial Health Score (0-100) for all companies.
    Uses 4 weighted dimensions with P10/P90 winsorisation.
    """
    df = df.copy()

    # Force all KPI columns to numeric first
    kpi_cols = [
        'return_on_equity_pct', 'return_on_capital_pct',
        'net_profit_margin_pct', 'cfo_quality_score',
        'free_cash_flow_cr', 'sales_cagr_5yr',
        'net_profit_cagr_5yr', 'debt_to_equity', 'interest_coverage'
    ]
    for col in kpi_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # ── Profitability Score (35%) ──────────────────────────────
    roe_w  = winsorise(df['return_on_equity_pct'].fillna(0))
    roce_w = winsorise(df['return_on_capital_pct'].fillna(0))
    npm_w  = winsorise(df['net_profit_margin_pct'].fillna(0))

    profitability_score = (
        scale_to_100(roe_w)  * 0.15 +
        scale_to_100(roce_w) * 0.10 +
        scale_to_100(npm_w)  * 0.10
    ) / 0.35 * 0.35

    # ── Cash Quality Score (30%) ───────────────────────────────
    cfo_w    = winsorise(df['cfo_quality_score'].fillna(0))
    fcf_flag = (df['free_cash_flow_cr'].fillna(0) > 0).astype(float) * 100

    cash_score = (
        scale_to_100(cfo_w) * 0.10 +
        fcf_flag            * 0.05
    ) / 0.15 * 0.30

    # ── Growth Score (20%) ─────────────────────────────────────
    rev_cagr_w = winsorise(df['sales_cagr_5yr'].fillna(0))
    pat_cagr_w = winsorise(df['net_profit_cagr_5yr'].fillna(0))

    growth_score = (
        scale_to_100(rev_cagr_w) * 0.10 +
        scale_to_100(pat_cagr_w) * 0.10
    ) / 0.20 * 0.20

    # ── Leverage Score (15%) ───────────────────────────────────
    de_w      = winsorise(df['debt_to_equity'].fillna(
        df['debt_to_equity'].median()))
    de_score  = (100 - scale_to_100(de_w))
    icr_w     = winsorise(df['interest_coverage'].fillna(0))
    icr_score = scale_to_100(icr_w)

    leverage_score = (
        de_score  * 0.10 +
        icr_score * 0.05
    ) / 0.15 * 0.15

    # ── Composite Score ────────────────────────────────────────
    df['health_score'] = (
        profitability_score +
        cash_score +
        growth_score +
        leverage_score
    ).round(1)

    df['health_score'] = df['health_score'].clip(0, 100)

    def get_band(score):
        if score >= 80:   return 'Excellent'
        elif score >= 65: return 'Good'
        elif score >= 50: return 'Average'
        elif score >= 35: return 'Weak'
        else:             return 'Poor'

    df['health_band'] = df['health_score'].apply(get_band)
    return df

screener_df = compute_health_score(screener_df)

print("Health Score Distribution:")
print(screener_df['health_band'].value_counts().to_string())
print()
print("Top 10 companies by Health Score:")
print(screener_df.nlargest(10, 'health_score')[
    ['company_id', 'broad_sector', 'health_score', 'health_band']
].to_string(index=False))

Health Score Distribution:
health_band
Average      40
Weak         29
Poor         15
Good         12
Excellent     2

Top 10 companies by Health Score:
company_id           broad_sector  health_score health_band
    INDIGO Consumer Discretionary          86.8   Excellent
       HAL            Industrials          80.8   Excellent
     IRCTC Consumer Discretionary          75.1        Good
 NESTLEIND       Consumer Staples          74.9        Good
       ABB            Industrials          73.3        Good
 COALINDIA              Materials          71.1        Good
     TRENT Consumer Discretionary          70.5        Good
      LICI             Financials          69.2        Good
       TCS Information Technology          68.3        Good
       BEL            Industrials          67.8        Good


In [5]:
# Re-run all presets now that screener_df has health scores
from src.screener.engine import apply_filters, load_config

config  = load_config()
presets = config['presets']

results = {}
for preset_name, preset_config in presets.items():
    filtered = apply_filters(screener_df, preset_config['filters'])
    filtered = filtered.sort_values('health_score', ascending=False)
    results[preset_name] = filtered
    print(f"  {preset_name:<25} → {len(filtered)} companies")

  quality_compounder        → 21 companies
  value_pick                → 7 companies
  growth_accelerator        → 19 companies
  dividend_champion         → 32 companies
  debt_free_blue_chip       → 21 companies
  turnaround_watch          → 36 companies


In [6]:
# Key columns to show in Excel
DISPLAY_COLS = [
    'company_id', 'broad_sector',
    'return_on_equity_pct', 'return_on_capital_pct',
    'net_profit_margin_pct', 'debt_to_equity',
    'interest_coverage', 'free_cash_flow_cr',
    'sales_cagr_5yr', 'net_profit_cagr_5yr',
    'pe_ratio', 'pb_ratio', 'dividend_yield_pct',
    'cfo_quality_score', 'asset_turnover',
    'health_score', 'health_band'
]

# Colour fills
GREEN  = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
RED    = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
HEADER = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')
GOLD   = PatternFill(start_color='FFD700', end_color='FFD700', fill_type='solid')

wb = Workbook()
wb.remove(wb.active)  # Remove default empty sheet

for preset_name, df in results.items():
    ws = wb.create_sheet(title=preset_name[:31])

    # Filter to display columns that exist
    cols = [c for c in DISPLAY_COLS if c in df.columns]
    df_display = df[cols].reset_index(drop=True)

    # Write header row
    ws.append(cols)
    for cell in ws[1]:
        cell.fill      = HEADER
        cell.font      = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center')

    # Write data rows
    for row in dataframe_to_rows(df_display, index=False, header=False):
        ws.append(row)

    # Colour code health score column
    hs_col = cols.index('health_score') + 1 if 'health_score' in cols else None
    if hs_col:
        for row_idx in range(2, ws.max_row + 1):
            cell  = ws.cell(row=row_idx, column=hs_col)
            score = cell.value
            if score is not None:
                if score >= 65:
                    cell.fill = GREEN
                elif score < 50:
                    cell.fill = RED

    # Auto-width columns
    for col in ws.columns:
        max_len = max(
            len(str(cell.value)) if cell.value else 0
            for cell in col
        )
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 25)

wb.save('output/screener_output.xlsx')
print("screener_output.xlsx saved to output/!")
print(f"Sheets created: {wb.sheetnames}")

screener_output.xlsx saved to output/!
Sheets created: ['quality_compounder', 'value_pick', 'growth_accelerator', 'dividend_champion', 'debt_free_blue_chip', 'turnaround_watch']


In [7]:
print("Health Score Summary:")
print(f"  Total companies scored: {screener_df['health_score'].notna().sum()}")
print()
print("Band Distribution:")
print(screener_df['health_band'].value_counts().to_string())
print()
print("Top 5 Healthiest Companies:")
print(screener_df.nlargest(5, 'health_score')[
    ['company_id', 'broad_sector', 'health_score', 'health_band']
].to_string(index=False))
print()
print("Bottom 5 Companies:")
print(screener_df.nsmallest(5, 'health_score')[
    ['company_id', 'broad_sector', 'health_score', 'health_band']
].to_string(index=False))

Health Score Summary:
  Total companies scored: 98

Band Distribution:
health_band
Average      40
Weak         29
Poor         15
Good         12
Excellent     2

Top 5 Healthiest Companies:
company_id           broad_sector  health_score health_band
    INDIGO Consumer Discretionary          86.8   Excellent
       HAL            Industrials          80.8   Excellent
     IRCTC Consumer Discretionary          75.1        Good
 NESTLEIND       Consumer Staples          74.9        Good
       ABB            Industrials          73.3        Good

Bottom 5 Companies:
company_id     broad_sector  health_score health_band
      BHEL      Industrials           9.5        Poor
  GODREJCP Consumer Staples          11.9        Poor
 TATASTEEL        Materials          20.7        Poor
       PFC       Financials          26.3        Poor
    RECLTD       Financials          27.6        Poor
